# Skin Lesion Classification on ISIC 2019
**EfficientNet-B0 · PyTorch · transfer learning · lesion-grouped validation**

This notebook fine-tunes an ImageNet-pretrained EfficientNet-B0 to classify dermoscopic images into **7 diagnostic categories**. The emphasis is on careful evaluation: a lesion-level split to prevent data leakage, class-balanced sampling for a heavily imbalanced dataset, and macro-averaged metrics so rare classes are not hidden behind the majority class.

> **Disclaimer:** educational / research project, not a medical device. Do not use it for diagnosis.

| | |
|---|---|
| **Dataset** | ISIC 2019 training set (colour-corrected with shades-of-gray), 7 classes used |
| **Model** | EfficientNet-B0, ImageNet weights, all layers fine-tuned |
| **Imbalance handling** | Inverse-frequency `WeightedRandomSampler` + macro-F1 model selection |
| **Validation** | 80 / 20 split grouped by `lesion_id` |
| **Hardware / time** | Kaggle Tesla T4 GPU, about 45 minutes for the full run |
| **Best validation macro-F1** | **0.582** (epoch 17), accuracy 0.684 |

### Contents
1. Setup and configuration
2. Data loading and label construction
3. Lesion-grouped train / validation split
4. Datasets, augmentation and class-balanced loading
5. Model
6. Training
7. Results
8. Extended evaluation (ROC-AUC, confusion matrix)
9. Inference on a single image
10. Notes, limitations and next steps

## 1. Setup and configuration

All paths and hyper-parameters live in one cell. If you are not on Kaggle, only the three paths need editing.

In [4]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

torch.backends.cudnn.benchmark = True  # auto-tune conv algorithms for the fixed input size
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [5]:
# ---------- Paths (edit if you are not running on Kaggle) ----------
IMG_DIR   = "/kaggle/input/isic-2019-prepocessed/step3_shadesofgray"
META_CSV  = "/kaggle/input/d/demirelmas/metadata-csv/metadata.csv"
GT_CSV    = "/kaggle/input/isic-2019-training-groundtruth/ISIC_2019_Training_GroundTruth.csv"
CKPT_PATH = "best_model.pt"

# ---------- Hyper-parameters ----------
IMG_SIZE     = 224                      # native EfficientNet-B0 input size
BATCH_SIZE   = 32
EPOCHS       = 25                       # one full cosine cycle
LR           = 3e-4                     # AdamW learning rate
NUM_WORKERS  = min(6, os.cpu_count() or 2)
VAL_FRACTION = 0.2
SEED         = 42                       # used for the train/val split

LABEL_COLS = ["MEL", "NV", "BCC", "DF", "AK", "BKL", "VASC"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

Device: cpu


## 2. Data loading and label construction

ISIC 2019 ships two CSVs that are joined on the image ID:

- `metadata.csv`: image IDs, lesion IDs and patient / lesion metadata.
- `ISIC_2019_Training_GroundTruth.csv`: one-hot columns for the diagnostic classes.

Processing steps:
1. **Merge** metadata and ground truth.
2. **Drop rows with missing values** (`dropna`).
3. **Keep only single-label rows** (exactly one positive class) to discard ambiguous annotations.
4. **Recover the class name** with `idxmax` over the one-hot columns.
5. **Map classes to integers** with an alphabetically sorted list, so the ordering is deterministic.

> Only 7 of the dataset's classes are used (`SCC` and `UNK` are not in `LABEL_COLS`), so images whose only positive label is one of those are removed by the single-label filter.

In [6]:
meta = pd.read_csv(META_CSV)
gt   = pd.read_csv(GT_CSV)

df = meta.merge(gt, on="image", how="inner")
df = df.dropna().reset_index(drop=True)
df = df[df[LABEL_COLS].sum(axis=1) == 1].reset_index(drop=True)
df["label"] = df[LABEL_COLS].idxmax(axis=1)

# Images without a lesion_id fall back to their own image ID, so every row has a group key.
# NOTE: dropna() above already removed rows with a missing lesion_id, so this line only
# matters if you move the fillna *before* the dropna.
df["lesion_id"] = df["lesion_id"].fillna(df["image"])

classes      = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label_idx"] = df["label"].map(class_to_idx)
assert df["label_idx"].isna().sum() == 0, "some labels failed to map"

print(f"{len(df):,} images | {df['lesion_id'].nunique():,} lesions | {len(classes)} classes")
print("Class index:", class_to_idx)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/d/demirelmas/metadata-csv/metadata.csv'

In [ ]:
counts = df["label"].value_counts()
ax = counts.plot.bar(figsize=(7, 3.2), color="tab:blue", rot=0)
ax.set_title("Class distribution (all filtered images)")
ax.set_ylabel("Images")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Lesion-grouped train / validation split

ISIC contains several images of the same physical lesion. A plain random split would put some in training and others in validation, rewarding the model for recognising a lesion it has effectively already seen and **inflating validation scores**.

`GroupShuffleSplit` with `lesion_id` as the group key puts **all images of a lesion in the same split**. The assertion below checks this explicitly.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(gss.split(df, groups=df["lesion_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

# No lesion may appear on both sides of the split
assert set(train_df["lesion_id"]).isdisjoint(val_df["lesion_id"]), "lesion leakage between splits!"

print(f"Train: {len(train_df):,} images | Validation: {len(val_df):,} images")
pd.concat({"train": train_df["label"].value_counts(),
           "val":   val_df["label"].value_counts()}, axis=1).loc[classes]

## 4. Datasets, augmentation and class-balanced loading

**Dataset.** A minimal `Dataset` that lazily loads one JPEG at a time (low RAM) and converts it to RGB.

**Augmentation (training only).**

| Transform | Why |
|---|---|
| `RandomResizedCrop` | simulates different zoom / framing of the lesion |
| `RandomHorizontalFlip` | lesions have no left/right orientation |
| `ColorJitter` (mild) | robustness to lighting and device differences |

**Validation** is deterministic (resize to 256, centre-crop 224). Both pipelines use ImageNet normalisation to match the pretrained backbone.

**Class imbalance.** Melanocytic nevi (`NV`) dominate the dataset. Each training sample gets weight `1 / class_count`, and a `WeightedRandomSampler` draws mini-batches with a roughly uniform class distribution (rare-class images are repeated within an epoch). The loss itself stays plain cross-entropy.

In [ ]:
class SkinDataset(Dataset):
    """Loads one dermoscopic image per item, lazily, and returns (tensor, class index)."""

    def __init__(self, frame, img_dir, transform):
        self.frame = frame
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['image']}.jpg")
        img = Image.open(path).convert("RGB")
        return self.transform(img), int(row["label_idx"])


train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
train_ds = SkinDataset(train_df, IMG_DIR, train_tfms)
val_ds   = SkinDataset(val_df,   IMG_DIR, val_tfms)

# Inverse-frequency sampling weights
class_counts   = train_df["label_idx"].value_counts().sort_index()
class_weights  = 1.0 / class_counts
sample_weights = train_df["label_idx"].map(class_weights).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,          # minority images may be drawn several times per epoch
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"),
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"{len(train_loader)} training batches | {len(val_loader)} validation batches")

## 5. Model

EfficientNet-B0 with ImageNet weights, with only the final linear layer replaced by a 7-way head. The backbone is **not frozen**: all layers are fine-tuned, which works well here because ISIC 2019 is moderately large (about 21k images after filtering). EfficientNet gives strong accuracy with far fewer parameters than ResNet / VGG, which keeps fine-tuning fast.

In [ ]:
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"EfficientNet-B0 | {n_params:.2f}M parameters | {len(classes)} output classes")

## 6. Training

| Component | Choice | Why |
|---|---|---|
| Loss | `CrossEntropyLoss` | standard for multi-class logits |
| Optimiser | `AdamW`, lr `3e-4` | decoupled weight decay; reliable for fine-tuning |
| Schedule | `CosineAnnealingLR`, `T_max = EPOCHS` | smooth decay of the learning rate to near zero |
| Precision | Automatic mixed precision | faster training and lower memory on GPU |
| Selection metric | **macro-F1** on validation | weights every class equally, unlike accuracy |

Each epoch trains on the balanced sampler, then evaluates on the validation set. The weights are saved whenever macro-F1 improves, so the final artefact is the best epoch, not the last. Lines marked `*` below are new best epochs.

> **Caveat:** the validation set is used both to pick the best epoch and to report results, so these numbers are slightly optimistic. See section 10.

In [ ]:
use_amp   = DEVICE == "cuda"
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler("cuda", enabled=use_amp)

In [ ]:
history = {"epoch": [], "train_loss": [], "val_macro_f1": [], "val_acc": []}
best_macro_f1, best_epoch, best_report, best_report_text = 0.0, 0, None, ""

for epoch in range(1, EPOCHS + 1):
    # ---------------- train ----------------
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            loss = criterion(model(x), y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    scheduler.step()
    train_loss = running_loss / len(train_loader)

    # --------------- validate ---------------
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            logits = model(x.to(DEVICE))
            preds.append(logits.argmax(1).cpu())
            targets.append(y)
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    report   = classification_report(targets, preds, target_names=classes, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]
    val_acc  = report["accuracy"]

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_macro_f1"].append(macro_f1)
    history["val_acc"].append(val_acc)

    improved = macro_f1 > best_macro_f1
    if improved:
        best_macro_f1, best_epoch, best_report = macro_f1, epoch, report
        best_report_text = classification_report(targets, preds, target_names=classes, digits=3)
        torch.save(model.state_dict(), CKPT_PATH)

    print(f"Epoch {epoch:>2}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"val macro-F1 {macro_f1:.4f} | val acc {val_acc:.3f}{'  *' if improved else ''}")

print(f"\nBest validation macro-F1: {best_macro_f1:.4f} (epoch {best_epoch}) -> saved to {CKPT_PATH}")
print(f"\nPer-class report at epoch {best_epoch}:\n{best_report_text}")

## 7. Results

Numbers below are from the logged run (validation split, 4,228 images). The checkpoint is the **epoch 17** model.

| Metric | Best epoch (17) | Final epoch (25) |
|---|---|---|
| Macro-F1 | **0.582** | 0.574 |
| Accuracy | 0.684 | 0.692 |

**What the curves show.** Training loss keeps falling while validation macro-F1 plateaus after roughly epoch 15 to 17, a sign of overfitting. Checkpointing on macro-F1 guards against taking the last epoch by default.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
ep = history["epoch"]

axes[0].plot(ep, history["train_loss"], marker="o", ms=3)
axes[0].set_title("Training loss")

axes[1].plot(ep, history["val_macro_f1"], marker="o", ms=3, color="tab:green")
axes[1].axvline(best_epoch, ls="--", color="grey", lw=1)
axes[1].set_title(f"Validation macro-F1  (best: epoch {best_epoch})")

axes[2].plot(ep, history["val_acc"], marker="o", ms=3, color="tab:orange")
axes[2].set_title("Validation accuracy")

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

**Per-class behaviour.** The majority class (`NV`) is easiest. Melanoma (`MEL`) recall is only about 0.54, which is too low for any clinical use, because missing a malignant lesion is the costly error. Rare classes (`DF`, n = 35; `VASC`, n = 57) have very few validation images, so their scores are noisy.

In [ ]:
metrics = ["precision", "recall", "f1-score"]
x = np.arange(len(classes))
w = 0.26

fig, ax = plt.subplots(figsize=(10, 4.2))
for i, m in enumerate(metrics):
    ax.bar(x + (i - 1) * w, [best_report[c][m] for c in classes], w, label=m)

ax.set_xticks(x)
ax.set_xticklabels([f"{c}\n(n={int(best_report[c]['support'])})" for c in classes])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title(f"Per-class validation metrics at the best epoch (epoch {best_epoch})")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Extended evaluation: ROC-AUC and confusion matrix

The training loop reports precision / recall / F1. This section adds **one-vs-rest ROC-AUC** and a **row-normalised confusion matrix** using the saved best checkpoint.

> These cells compute metrics on the **validation** split, which was also used for model selection. For unbiased numbers, create a separate test split and evaluate on that once (see section 10). Run this section after training; it has not been executed in the saved notebook.

In [ ]:
# Reload the best checkpoint and collect class probabilities on the validation set
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for x, y in val_loader:
        logits = model(x.to(DEVICE))
        all_probs.append(torch.softmax(logits.float(), dim=1).cpu())
        all_targets.append(y)

probs   = torch.cat(all_probs).numpy()
targets = torch.cat(all_targets).numpy()
preds   = probs.argmax(1)

macro_auc = roc_auc_score(targets, probs, multi_class="ovr", average="macro")
per_class_auc = {c: roc_auc_score((targets == i).astype(int), probs[:, i])
                 for i, c in enumerate(classes)}

print(f"Macro one-vs-rest ROC-AUC: {macro_auc:.3f}\n")
print(pd.Series(per_class_auc, name="ROC-AUC").round(3).to_string())

In [ ]:
cm = confusion_matrix(targets, preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)      # each row sums to 1 (= recall per class)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes)
ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalised)")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 9. Inference on a single image

Loads the saved weights and returns class probabilities for one image, using the same deterministic preprocessing as validation.

In [ ]:
@torch.no_grad()
def predict(image_path, top_k=3):
    """Return the top-k (class, probability) pairs for one dermoscopic image."""
    img = Image.open(image_path).convert("RGB")
    x = val_tfms(img).unsqueeze(0).to(DEVICE)
    model.eval()
    p = torch.softmax(model(x).float(), dim=1)[0].cpu()
    top = torch.topk(p, k=min(top_k, len(classes)))
    return [(classes[i], round(float(v), 3)) for v, i in zip(top.values, top.indices)]


# Example (uncomment and point to a real image):
predict(os.path.join(IMG_DIR, val_df.loc[0, "image"] + ".jpg"))

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## 10. Notes, limitations and next steps

**Limitations**
- **No separate test set.** The validation split picks the best epoch and also reports the results, so scores are optimistically biased. A cleaner protocol uses train / validation / test (or grouped cross-validation) and evaluates on the test set once.
- **Single run, no variance estimate.** No global random seeds are set (only the split is seeded) and `cudnn.benchmark` is on, so reruns will differ slightly. Differences of a few points, especially for `DF` (n = 35) and `VASC` (n = 57), are within plausible noise.
- **Seven classes only, with filtering.** `SCC` and `UNK` are not modelled. `dropna` removes every row with any missing metadata value (for example age, sex or anatomical site), which can introduce selection bias because missingness is not random. It also runs before the `lesion_id` fallback, so images lacking a `lesion_id` are dropped rather than kept.
- **Overfitting.** Training loss keeps falling while validation macro-F1 plateaus after roughly epoch 15 to 17.
- **Dataset bias.** ISIC data come from specific institutions, devices and (predominantly lighter-skinned) patient populations, so performance may not generalise.
- **Not clinically usable.** Melanoma recall of about 0.54 is far too low.

**Next steps**
- Add a held-out test split and report ROC-AUC, confusion matrices and calibration on it.
- Compare class-weighted loss or focal loss against sampler-based oversampling, or combine them.
- Stronger augmentation (rotations, vertical flips, MixUp / CutMix) and larger backbones or resolutions.
- Include `SCC` and treat `UNK` as out-of-distribution; use patient metadata (age, sex, site) in a multimodal model.
- Several seeds or grouped k-fold cross-validation, reporting mean and standard deviation.
- Grad-CAM to verify the model looks at the lesion rather than artefacts such as rulers or ink marks.
- Tune thresholds for high melanoma sensitivity and report sensitivity at a fixed specificity.